In [1]:
import json

json_path = "../data/hadith_extraction_clean.json"

with open(json_path, "r", encoding="utf-8") as f:
    hadiths = json.load(f)

print("Number of hadiths:", len(hadiths))

Number of hadiths: 42


In [2]:
hadiths[0]

In [3]:
documents = []

for hadith in hadiths:
    document = {
        "content": f"{hadith['title']}\n\n{hadith['hadith_text']}",
        "metadata": {
            "hadith_number": hadith["hadith_number"],
            "title": hadith["title"],
            "type": "hadith",
            "narrator": hadith["narrator"],
            "pages": hadith["pages"],
        }
    }

    documents.append(document)

print("Number of documents:", len(documents))

Number of documents: 42


In [4]:
print("CONTENT:")
print(documents[0]["content"])

print("\nMETADATA:")
documents[0]["metadata"]

CONTENT:
الأعمال بالنيات

عن أمير المؤمنين أبي حفص عمر بن الخطاب قال: سمعت رسول الله ﷺ يقول: " إنما الأعمال بالنيات، وإنما لكل امرىء ما نوى، فمن كانت هجرته إلى الله ورسوله فهجرته إلى الله ورسوله، ومن كانت هجرته لدنيا يصيبها، أو امرأة ينكحها، فهجرته إلى ما هاجر إليه " رواه إماما المحدثين أبو عبدالله محمد بن إسماعيل بن إبراهيم بن المغيرة بن بردزبه البخاري، وأبو الحسين مسلم بن الحجاج ين مسلم القشيري النيسابوري، في صحيحيهما اللذين هما أصح الكتب المصنفة.

METADATA:


In [5]:
for hadith in hadiths:
    if hadith["sharh"].strip():
        document = {
            "content": f"{hadith['title']}\n\n{hadith['sharh']}",
            "metadata": {
                "hadith_number": hadith["hadith_number"],
                "title": hadith["title"],
                "type": "sharh",
                "pages": hadith["pages"],
            }
        }

        documents.append(document)

print("Total documents:", len(documents))

Total documents: 84


In [6]:
documents[42]

In [7]:
rawi_documents = {}

for hadith in hadiths:
    narrator = hadith["narrator"].strip()
    rawi_bio = hadith["rawi_bio"].strip()

    if not narrator or not rawi_bio:
        continue

    if narrator not in rawi_documents:
        rawi_documents[narrator] = {
            "content": f"{narrator}\n\n{rawi_bio}",
            "metadata": {
                "title": "راوي الحديث",
                "type": "rawi_bio",
                "hadith_numbers": [hadith["hadith_number"]],
            }
        }
    else:
        rawi_documents[narrator]["metadata"]["hadith_numbers"].append(
            hadith["hadith_number"]
        )

documents.extend(rawi_documents.values())

print("Unique narrators:", len(rawi_documents))
print("Total documents:", len(documents))

Unique narrators: 32
Total documents: 116


In [8]:
documents[84]

In [9]:
from collections import Counter

type_counts = Counter(
    doc["metadata"]["type"]
    for doc in documents
)

print(type_counts)

Counter({'hadith': 42, 'sharh': 42, 'rawi_bio': 32})


In [10]:
for doc in documents[:3]:
    print(doc["metadata"])

{'hadith_number': 1, 'title': 'الأعمال بالنيات', 'type': 'hadith', 'narrator': 'أمير المؤمنين أبي حفص عمر بن الخطاب ﴿رضي الله تعالى عنه﴾', 'pages': [3]}
{'hadith_number': 2, 'title': 'مراتب الدين', 'type': 'hadith', 'narrator': 'عمر ﴿رضي الله تعالى عنه﴾', 'pages': [4, 5]}
{'hadith_number': 3, 'title': 'أركان الإسلام', 'type': 'hadith', 'narrator': 'أبي عبد الرحمن عبد الله بن عمر بن الخطاب ﴿رضي الله تعالى عنه﴾', 'pages': [5]}


In [11]:
empty_docs = [
    i for i, doc in enumerate(documents)
    if not doc["content"].strip()
]

print("Empty documents:", len(empty_docs))

Empty documents: 0


In [12]:
for doc_type in ["hadith", "sharh", "rawi_bio"]:
    lengths = [
        len(doc["content"])
        for doc in documents
        if doc["metadata"]["type"] == doc_type
    ]

    print(f"\n{doc_type}")
    print("Number of documents:", len(lengths))
    print("Min length:", min(lengths))
    print("Max length:", max(lengths))
    print("Average length:", round(sum(lengths) / len(lengths), 2))


hadith
Number of documents: 42
Min length: 98
Max length: 963
Average length: 331.43

sharh
Number of documents: 42
Min length: 152
Max length: 6700
Average length: 1889.79

rawi_bio
Number of documents: 32
Min length: 479
Max length: 1940
Average length: 1015.91


In [13]:
import json

documents_path = "../data/documents.json"

with open(documents_path, "w", encoding="utf-8") as f:
    json.dump(documents, f, ensure_ascii=False, indent=2)

print(f"Saved {len(documents)} documents to {documents_path}")

Saved 116 documents to ../data/documents.json
